In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

!git clone https://github.com/MgSO4-7H2O/AI_safety_lab.git
%cd /content/AI_safety_lab/lab2/adv_cv
!pwd
!ls

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)

        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)

        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)

        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)

        x = torch.flatten(x, 1)

        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)

        x = self.fc2(x)

        output = F.log_softmax(x, dim=1)
        return output

In [ ]:
class MobileNet(nn.Module):
    def __init__(self):
        super(MobileNet, self).__init__()

        base_model = models.mobilenet_v3_small(weights=None)

        self.trunk = nn.Sequential(
            base_model.features,
            base_model.avgpool,
            nn.Flatten()
        )

        self.fc = nn.Sequential(
            nn.Linear(576, 10)
        )

    def forward(self, x):
        x = self.trunk(x)
        x = self.fc(x)
        output = F.log_softmax(x, dim=1)
        return output


print("Models defined.")

In [ ]:
def test(model, device, test_loader, name="model"):
    model.eval()
    test_loss = 0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)
            target = target.to(device)

            output = model(data)
            test_loss += F.nll_loss(output, target, reduction="sum").item()

            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)

    print(f"{name}")
    print(f"Average loss: {test_loss:.4f}")
    print(f"Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)")
    print()

    return accuracy

In [ ]:
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.1307, 0.1307, 0.1307),
        (0.3081, 0.3081, 0.3081)
    )
])

test_dataset = datasets.ImageFolder(
    root="mnist/testing",
    transform=test_transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=100,
    shuffle=False,
    num_workers=0
)

print("Test dataset size:", len(test_dataset))
print("Classes:", test_dataset.classes)

In [ ]:
cnn = Net().to(device)
cnn.load_state_dict(torch.load("mnist_cnn.pt", map_location=device))
cnn.eval()

mobile = MobileNet().to(device)
mobile.load_state_dict(torch.load("mnist_mobile.pt", map_location=device))
mobile.eval()

print("CNN and MobileNet loaded.")

In [ ]:
cnn_clean_acc = test(cnn, device, test_loader, name="CNN on clean MNIST")
mobile_clean_acc = test(mobile, device, test_loader, name="MobileNet on clean MNIST")